# Repo Filter

Generates a self-contained HTML page with Plotly parallel coordinates
and AG Grid table. Works entirely client-side — no Python backend needed.

The HTML is displayed inline and can be saved to `repo_filter_out.html`.

### Prerequisites

```bash
# Fetch + enrich + merge
export GITHUB_TOKEN=ghp_...
uv run python evals/eda/fetch_repos.py
uv run python evals/eda/enrich_test_counts.py
uv run python evals/eda/merge_repo_lists.py
```


In [ ]:
import json as _json
from pathlib import Path

from IPython.display import HTML, display


In [ ]:
# ── Configuration ──
# Which JSONL to load (merged list by default, or filtered/selected)
JSONL_CANDIDATES = [
    "merged_repo_list_proposal.jsonl",
    "evals/eda/merged_repo_list_proposal.jsonl",
    "filtered_repos.jsonl",
    "evals/eda/filtered_repos.jsonl",
    "selected_repos.jsonl",
    "evals/eda/selected_repos.jsonl",
]

HTML_TEMPLATE_CANDIDATES = [
    "repo_filter.html",
    "evals/eda/repo_filter.html",
]

OUTPUT_HTML = Path("repo_filter_out.html")


In [ ]:
# ── Load data ──
jsonl_path = None
for candidate in JSONL_CANDIDATES:
    p = Path(candidate)
    if p.exists():
        jsonl_path = p
        break
if jsonl_path is None:
    raise FileNotFoundError("No JSONL data file found. Run the pipeline scripts first.")

records = [_json.loads(line) for line in jsonl_path.read_text().splitlines() if line.strip()]
print(f"Loaded {len(records)} repos from {jsonl_path}")

# ── Load HTML template ──
template_path = None
for candidate in HTML_TEMPLATE_CANDIDATES:
    p = Path(candidate)
    if p.exists():
        template_path = p
        break
if template_path is None:
    raise FileNotFoundError("repo_filter.html template not found")

template = template_path.read_text()

# ── Inject data into template ──
data_json = _json.dumps(records)
html_out = template.replace('"__DATA_PLACEHOLDER__"', data_json)

# ── Save to file ──
OUTPUT_HTML.write_text(html_out)
print(f"Saved self-contained HTML to {OUTPUT_HTML} ({len(html_out) // 1024} KB)")
print(f"Open in browser: file://{OUTPUT_HTML.resolve()}")


In [ ]:
# ── Display inline (in an iframe to isolate CSS/JS) ──
import base64 as _b64

_data_uri = "data:text/html;base64," + _b64.b64encode(html_out.encode()).decode()
display(HTML(
    f'<iframe src="{_data_uri}" '
    f'width="100%" height="1250" style="border:1px solid #ddd; border-radius:4px"></iframe>'
))
